In [22]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from numpy import mean
from numpy import absolute
from numpy import sqrt
import pandas as pd

In [24]:
df = pd.DataFrame({'y': [6, 8, 12, 14, 14, 15, 17, 22, 24, 23],
                   'x1': [2, 5, 4, 3, 4, 6, 7, 5, 8, 9],
                   'x2': [14, 12, 12, 13, 7, 8, 7, 4, 6, 5]})

In [26]:
#define predictor and response variables
X = df[['x1', 'x2']]
y = df['y']

#define cross-validation method to use
cv = KFold(n_splits=10, random_state=1, shuffle=True)

#build multiple linear regression model
model = LinearRegression()

#use k-fold CV to evaluate model
scores = cross_val_score(model, X, y, scoring='neg_mean_absolute_error',
                         cv=cv, n_jobs=-1)

#view mean absolute error
mean(absolute(scores))

3.1461548083469744

In [32]:
from sklearn.tree import DecisionTreeRegressor # Changed to Regressor
from sklearn.model_selection import train_test_split

# Define X and y from the DataFrame 'df' (must be run before this cell)
X = df[['x1', 'x2']].to_numpy()
y = df['y'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state = 4)

model = DecisionTreeRegressor() # Use Regressor
model.fit(X_train, y_train)

# For regression, we typically look at R^2 score.
from sklearn.metrics import r2_score
result = r2_score(y_test, model.predict(X_test))

print("R^2 Score:", result)

R^2 Score: -0.30000000000000004


In [60]:
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import pandas as pd # Ensure pandas is imported if running this cell alone

# Define X and y from the DataFrame 'df' (Assuming df is defined in Cell In[2])
X = df.drop(columns=['y']).to_numpy()
y = df['y'].to_numpy()

pipeline = make_pipeline(
    StandardScaler(),
    RandomForestRegressor(n_estimators=100, max_depth=4, random_state=42)
)

# Split the data (n_train = 8)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=4
)

# Use cv=5 because the training set size is 8. 
# cv must be <= 8.
scores = cross_val_score(pipeline, X_train, y_train, cv=5, n_jobs=1, scoring='neg_mean_squared_error')

# Print the Mean Squared Error (MSE) and its standard deviation
print('Cross Validation MSE scores (negated): %s' % scores)
print('Cross Validation Root Mean Squared Error (RMSE): %.3f +/- %.3f' % (np.sqrt(np.mean(np.absolute(scores))), np.std(np.sqrt(np.absolute(scores)))))

Cross Validation MSE scores (negated): [-12.85505  -3.40405 -37.82825  -2.4025  -43.8244 ]
Cross Validation Root Mean Squared Error (RMSE): 4.479 +/- 2.112


In [62]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

In [64]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state = 4)

model = DecisionTreeClassifier()

model.fit(X_train,  y_train)
result = model.score(X_test, y_test)

print(result)

0.0


In [74]:
import numpy as np
from sklearn.linear_model import Lasso, LassoCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
import pandas as pd # Ensure pandas is imported

# 1. Define X and y from the DataFrame 'df' (Assuming df is defined in Cell In[2])
X = df[['x1', 'x2']].to_numpy() 
y = df['y'].to_numpy()

# 2. Split the data (n_train = 8)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=4)

# FIX: Changed cv from 10 to 5 since n_train is only 8
pipe_cv = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso_cv", LassoCV(cv=5, n_alphas=1000, max_iter=10000, random_state=42))
])
pipe_cv.fit(X_train, y_train)
best_alpha = pipe_cv.named_steps["lasso_cv"].alpha_

lasso_best = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", Lasso(alpha=best_alpha, max_iter=10000))
])
lasso_best.fit(X_train, y_train)

coef = lasso_best.named_steps["lasso"].coef_
intercept = lasso_best.named_steps["lasso"].intercept_

# Convert the array index of the coefficients back to the feature name
feature_names = ['x1', 'x2']
pairs = list(zip(coef, feature_names)) 
pairs_sorted = sorted(pairs, key=lambda t: abs(t[0]), reverse=True)

print("alpha:", best_alpha)
print("intercept:", intercept)
print("r2_train:", round(r2_score(y_train, lasso_best.predict(X_train)), 4))
print("r2_test:", round(r2_score(y_test, lasso_best.predict(X_test)), 4))
print("mse_test:", mean_squared_error(y_test, lasso_best.predict(X_test)))

print("top coefficients:", pairs_sorted)
zero = [name for c, name in pairs if np.isclose(c, 0.0)]
neg = [name for c, name in pairs if c < 0 and not np.isclose(c, 0.0)]
pos = [name for c, name in pairs if c > 0 and not np.isclose(c, 0.0)]
print("ineffective:", zero)
print("negative:", neg)
print("positive:", pos)

alpha: 0.10762220022583623
intercept: 14.625
r2_train: 0.9205
r2_test: -0.2054
mse_test: 30.1353981382165
top coefficients: [(-4.472974019517924, 'x2'), (1.1463031328495992, 'x1')]
ineffective: []
negative: ['x2']
positive: ['x1']
